# Исследование надёжности заёмщиков

Заказчик — кредитный отдел банка. Нужно разобраться, влияет ли семейное положение и количество детей клиента на факт погашения кредита в срок. Входные данные от банка — статистика о платёжеспособности клиентов.

Результаты исследования будут учтены при построении модели **кредитного скоринга** — специальной системы, которая оценивает способность потенциального заёмщика вернуть кредит банку.

## Шаг 1. Откройте файл с данными и изучите общую информацию

In [148]:
import pandas as pd
from IPython.display import display

data = pd.read_csv('data.csv')
display(data.head())
data.info()

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


<a id='general_info'>**Вывод**</a>

В таблице 12 столбцов и 21525 строчел. В каждой строчки таблицы - данные о клиенте.

Согласно документации к данным:
* `children` — количество детей в семье;
* `days_employed` — общий трудовой стаж в днях;
* `dob_years` — возраст клиента в годах;
* `education` — уровень образования клиента;
* `education_id` — идентификатор уровня образования;
* `family_status` — семейное положение;
* `family_status_id` — идентификатор семейного положения;
* `gender` — пол клиента;
* `income_type` — тип занятости;
* `debt` — имел ли задолженность по возврату кредитов;
* `total_income` — ежемесячный доход;
* `purpose` — цель получения кредита.

В столбцах `total_income` и `days_employed` встречаются пропуски. Также тип в этих столбцах `float`, хотя по природе данных (количество дней и доход) должен быть `int`. Типы данных других столбцов отображают действительность.
В `days_emploted` можно увидеть много отрицательных значений, это также артефакт.

## Шаг 2. Предобработка данных

### Обработка пропусков

Сначала посчитаем количество пропущенных значений в таблице.

In [149]:
data.isna().sum()

children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64

Как видим, количество пропущенных значений в столбцах `days_employed` и `total_income` одинаково, и встречаются пропуски у одних и тех же клиентов. Скорее всего, у источника информации для этих клиентов просто не было данных про стаж и доход.
Столбец `days_employed` никак не влияет на проверку гипотез и конечный результат, поэтому мы можем его удалить или заполнить пропуски произвольным значением, например нулем.

In [150]:
data['days_employed'] = data['days_employed'].fillna(0)

Но пропуски в `total_income` могут исказить конечный результат. Лучшим вариантом устранить пропуски было бы уточнить информацию у разработчика, который дал нам эти данные, но такой возможности у нас нет. Поэтому чтобы заполнить пропущенные значения, мы посчитаем медиану для каждой группы по типу занятости и запишем соответствующие значения для каждого клиента, относительно его `income_type`.

In [151]:
data['total_income'] = data.groupby(['income_type'])['total_income'].apply(
    lambda income: income.fillna(income.median())
)

Убедимся, что в таблице не осталось пропусков

In [152]:
data.isna().sum()

children            0
days_employed       0
dob_years           0
education           0
education_id        0
family_status       0
family_status_id    0
gender              0
income_type         0
debt                0
total_income        0
purpose             0
dtype: int64

### Замена типа данных

Как мы уже выяснили ранее, столбцы `total_income` и `days_employed` распознаны как `float`. Это означает, что в столбцах есть дробные значения, что противоречит природе данных. Мы не можем точно знать причину такого типа, но скорее всего это произошло в результате деления.

Например, для `days_employed` могли произойти следующие вычисления:
`secs_employed = end_of_work - start_of_work`, где `end_of_work` и `start_of_work` время конца и начало работы в формате **unix time**
`days_employed = secs_employed / 60 / 60 / 24` - преобразование числа из секунд в дни

Поэтому для более удобной работы в дальнейшем, превратим данные `total_income`, `days_employed` в целочисленные с помощью метода `astype()`.

In [153]:
data['total_income'] = data['total_income'].astype('int')
data['days_employed'] = data['days_employed'].astype('int')

Проверим полученный результат, вызвав метод `info()`

In [154]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   children          21525 non-null  int64 
 1   days_employed     21525 non-null  int64 
 2   dob_years         21525 non-null  int64 
 3   education         21525 non-null  object
 4   education_id      21525 non-null  int64 
 5   family_status     21525 non-null  object
 6   family_status_id  21525 non-null  int64 
 7   gender            21525 non-null  object
 8   income_type       21525 non-null  object
 9   debt              21525 non-null  int64 
 10  total_income      21525 non-null  int64 
 11  purpose           21525 non-null  object
dtypes: int64(7), object(5)
memory usage: 2.0+ MB


Видно, что в столбцах `total_income` и `days_employed` сохраняются только целочисленные значения.

Когда мы узучали [общую информацию](#general_info) о таблице, то заметили, что в столбце много отрицательных значений. Можем предположить, что они возникли в результате неправильной перестановки операндов при вычитании (`start_of_work - end_of_work`). Для того чтобы данные отражали действительность, превратим отрицательные значения в положительные с помощью функции `abs()`.

In [155]:
data['days_employed'] = abs(data['days_employed'])

### Обработка дубликатов

In [156]:
data.duplicated().sum()

54

С поможу метода `drop_duplicates()` удалим явные дубликаты с удалением старых индексов и формированием новых

In [157]:
data = data.drop_duplicates().reset_index(drop=True)

Убедимся, что мы полностью избавились от явных дубликатов

In [158]:
data.duplicated().sum()

0

Теперь попробуем отыскать неявные дубликаты в каждом столбце, где их наличие вероятно:

In [159]:
print(
    data['education'].value_counts(),
    data['family_status'].value_counts(),
    data['income_type'].value_counts(),
    data['purpose'].value_counts(),
    sep='\n\n'
)

среднее                13705
высшее                  4710
СРЕДНЕЕ                  772
Среднее                  711
неоконченное высшее      668
ВЫСШЕЕ                   273
Высшее                   268
начальное                250
Неоконченное высшее       47
НЕОКОНЧЕННОЕ ВЫСШЕЕ       29
НАЧАЛЬНОЕ                 17
Начальное                 15
ученая степень             4
Ученая степень             1
УЧЕНАЯ СТЕПЕНЬ             1
Name: education, dtype: int64

женат / замужем          12344
гражданский брак          4163
Не женат / не замужем     2810
в разводе                 1195
вдовец / вдова             959
Name: family_status, dtype: int64

сотрудник          11091
компаньон           5080
пенсионер           3837
госслужащий         1457
безработный            2
предприниматель        2
студент                1
в декрете              1
Name: income_type, dtype: int64

свадьба                                   793
на проведение свадьбы                     773
сыграть свадьбу    

Видно, что в столбце `education` много неявных дубликатов, отличающихся регистром. А в столбце `pursope` одни и те же цели получения кредита записаны в разных формах. Во всех остальных столбцах значения уникальны.
Чтобы избавиться от дубликатов в `education` приведем все значения к нижнему регистру с помощью метода `str.lower()`. Избавление от дубликатов требует более сложного процесса – [**лемматизации**](#lemmatisation).

In [160]:
data['education'] = data['education'].str.lower()

Проверим, появились ли дубликаты в таблице после наших преобразований.

In [161]:
data.duplicated().sum()

17

С поможу метода `drop_duplicates()` избавимся от дубликатов. А после этого проверим удалили ли мы их всех.


In [162]:
data = data.drop_duplicates().reset_index(drop=True)
data.duplicated().sum()

0

Дубликатов в таблице нет, можем переходить к следующему шагу – лемматизации.

### <a id='lemmatisation'>Лемматизация</a>

In [173]:
from pymystem3 import Mystem
m = Mystem()

purpose_categories = {
        'коммерческий': 'коммерция',
        'жилье': 'недвижимость',
        'недвижимость': 'недвижимость',
        'свадьба': 'свадьба',
        'автомобиль': 'автомобиль',
        'образование': 'образование'
    }

def get_purpose_category(purpose):
    lemmas = m.lemmatize(purpose)
    for key, value in purpose_categories.items():
        if key in lemmas:
            return value

data['purpose_category'] = data['purpose'].apply(get_purpose_category)

display(data)
data[data['purpose_category'] == 'коммерция'].count()

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose,purpose_category
0,1,8437,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875,покупка жилья,недвижимость
1,1,4024,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080,приобретение автомобиля,автомобиль
2,0,5623,33,среднее,1,женат / замужем,0,M,сотрудник,0,145885,покупка жилья,недвижимость
3,3,4124,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628,дополнительное образование,образование
4,0,340266,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616,сыграть свадьбу,свадьба
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21449,1,4529,43,среднее,1,гражданский брак,1,F,компаньон,0,224791,операции с жильем,недвижимость
21450,0,343937,67,среднее,1,женат / замужем,0,F,пенсионер,0,155999,сделка с автомобилем,автомобиль
21451,1,2113,38,среднее,1,гражданский брак,1,M,сотрудник,1,89672,недвижимость,недвижимость
21452,3,3112,38,среднее,1,женат / замужем,0,M,сотрудник,1,244093,на покупку своего автомобиля,автомобиль


children            1311
days_employed       1311
dob_years           1311
education           1311
education_id        1311
family_status       1311
family_status_id    1311
gender              1311
income_type         1311
debt                1311
total_income        1311
purpose             1311
purpose_category    1311
dtype: int64